# Train LSTM on Colab

This notebook sets up Colab, copies `data/lstm_processed/lstm_v1` to local disk, and launches the reusable source training script `src/training/lstm_training/train_lstm.py`.

## 1. Mount Drive

In [3]:
%cd /content
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Paths

In [4]:
from pathlib import Path
import os
import shutil

# Keep code on local Colab disk so pip/python never run from a fragile Drive FUSE cwd.
# Copy only runnable repo files; Google Drive virtual files like *.gdoc cannot be copied by shutil.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Seoul_bike_project")
CODE_DIR = Path("/content/Seoul_bike_project")
RELOAD_LOCAL_CODE = True

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f"Drive project directory does not exist: {DRIVE_PROJECT_DIR}")

if CODE_DIR.exists() and RELOAD_LOCAL_CODE:
    shutil.rmtree(CODE_DIR)

if not CODE_DIR.exists():
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    for dirname in ["src", "configs"]:
        src = DRIVE_PROJECT_DIR / dirname
        dst = CODE_DIR / dirname
        if not src.exists():
            raise FileNotFoundError(f"Missing required repo directory: {src}")
        shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints"))
    for filename in ["requirements.txt", "README.md"]:
        src = DRIVE_PROJECT_DIR / filename
        if src.exists():
            shutil.copy2(src, CODE_DIR / filename)

PROJECT_DIR = CODE_DIR
DATASET_NAME = "lstm_v1"
DRIVE_DATA_DIR = DRIVE_PROJECT_DIR / "data" / "lstm_processed" / DATASET_NAME
LOCAL_DATA_DIR = Path("/content/lstm_processed") / DATASET_NAME
DATA_DIR = LOCAL_DATA_DIR

CHECKPOINT_DIR = DRIVE_PROJECT_DIR / "checkpoints" / "lstm_v1"
MODEL_DIR = DRIVE_PROJECT_DIR / "models" / "lstm_v1"
LOG_DIR = DRIVE_PROJECT_DIR / "logs" / "lstm_v1"

os.chdir(PROJECT_DIR)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Code directory:", PROJECT_DIR)
print("Drive project:", DRIVE_PROJECT_DIR)
print("Drive dataset:", DRIVE_DATA_DIR)
print("Local dataset:", DATA_DIR)
print("Current working directory:", Path.cwd())

Project: /content/drive/MyDrive/Seoul_bike_project
Drive dataset: /content/drive/MyDrive/Seoul_bike_project/data/lstm_processed/lstm_v1
Local dataset: /content/lstm_processed/lstm_v1


## 3. Install Requirements

In [6]:
%cd /content/Seoul_bike_project
!python -m pip install -r requirements.txt

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

## 4. Copy Dataset to Colab Disk

In [ ]:
import shutil

RELOAD_LOCAL_DATA = True
if not DRIVE_DATA_DIR.exists():
    raise FileNotFoundError(f"Missing dataset directory: {DRIVE_DATA_DIR}")
if LOCAL_DATA_DIR.exists() and RELOAD_LOCAL_DATA:
    shutil.rmtree(LOCAL_DATA_DIR)
if not LOCAL_DATA_DIR.exists():
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
print("Using local dataset:", DATA_DIR)

## 5. Verify Dataset

In [ ]:
import json

required = [
    DATA_DIR / "dynamic_features.npy",
    DATA_DIR / "targets.npy",
    DATA_DIR / "targets_raw.npy",
    DATA_DIR / "static_numeric.npy",
    DATA_DIR / "sample_index_train.npy",
    DATA_DIR / "sample_index_val.npy",
    DATA_DIR / "feature_config.json",
    DATA_DIR / "dataset_summary.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing dataset files:\n" + "\n".join(missing))
with open(DATA_DIR / "dataset_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print(json.dumps({
    "T_total": summary["T_total"],
    "S": summary["S"],
    "samples_per_split": summary["samples_per_split"],
}, indent=2))

## 6. Train With Source Script

In [1]:
!python -m src.training.lstm_training.train_lstm \
  --config configs/lstm_baseline.yaml \
  --data_dir /content/lstm_processed/lstm_v1 \
  --checkpoint_dir /content/drive/MyDrive/Seoul_bike_project/checkpoints/lstm_v1 \
  --model_dir /content/drive/MyDrive/Seoul_bike_project/models/lstm_v1 \
  --log_dir /content/drive/MyDrive/Seoul_bike_project/logs/lstm_v1 \
  --batch_size 32768 \
  --resume auto \
  --resume_mode auto \
  --wandb_enabled true \
  --smoke_test true

/usr/bin/python3: Error while finding module specification for 'src.training.lstm_training.train_lstm' (ModuleNotFoundError: No module named 'src')
